In [2]:
import cv2
import numpy as np
from sklearn.metrics import confusion_matrix, matthews_corrcoef
from scipy.spatial.distance import directed_hausdorff
from scipy import ndimage
from tensorflow.keras.models import load_model
import os
import pandas as pd

# Path to the saved model
model_path = r"E:\ASI\ASI_Retinal_ChaseGT1.h5"
model = load_model(model_path, compile=False)
print("Model loaded successfully.")

# Define paths
gray_dir = r"E:\ASI\CHASE DB1\Chase\Images"
rvs_dir = r"E:\ASI\CHASE DB1\Chase\GT1"

# Initialize metrics storage with image names
metrics = {
    'RVS': {'Image': [], 'SEN': [], 'SPC': [], 'PRE': [], 'ACC': [], 'JAC': [], 'DICE': [], 'H1': [], 'H2': [],
            'ASSD': [], 'MCC': [], 'FMI': [], 'Brier_Score': []}
}

def calculate_all_metrics(gt_mask, pred_mask, pred_prob):
    """Calculate comprehensive segmentation metrics."""
    # Ensure binary masks
    gt_binary = (gt_mask > 0.5).astype(np.uint8)
    pred_binary = (pred_mask > 0.5).astype(np.uint8)
    
    # Flatten arrays for metric calculations
    gt_flat = gt_binary.flatten()
    pred_flat = pred_binary.flatten()
    prob_flat = pred_prob.flatten()
    
    # Handle edge cases
    if np.sum(gt_flat) == 0 and np.sum(pred_flat) == 0:
        # Both masks are empty - perfect match
        tn = len(gt_flat)
        fp = fn = tp = 0
    elif np.sum(gt_flat) == 0:
        # Ground truth is empty but prediction is not
        tn = np.sum(pred_flat == 0)
        fp = np.sum(pred_flat == 1)
        fn = tp = 0
    else:
        # Standard case
        tn, fp, fn, tp = confusion_matrix(gt_flat, pred_flat).ravel()
    
    # Calculate basic metrics
    eps = 1e-10  # Avoid division by zero
    
    # Sensitivity/Recall
    sen = tp / (tp + fn + eps)
    
    # Specificity
    spc = tn / (tn + fp + eps)
    
    # Precision
    pre = tp / (tp + fp + eps)
    
    # Accuracy
    acc = (tp + tn) / (tp + tn + fp + fn + eps)
    
    # Dice Coefficient (F1 Score)
    dice = (2 * tp) / (2 * tp + fp + fn + eps)
    
    # Jaccard Index (IoU)
    jac = tp / (tp + fp + fn + eps)
    
    # Matthews Correlation Coefficient (MCC)
    mcc = matthews_corrcoef(gt_flat, pred_flat)
    
    # Fowlkes-Mallows Index (FMI)
    fmi = tp / np.sqrt((tp + fp) * (tp + fn) + eps)
    
    # Brier Score (Probability calibration)
    brier_score = np.mean((prob_flat - gt_flat) ** 2)
    
    # ASSD
    assd_metrics = calculate_assd(gt_mask, pred_mask)
    assd = assd_metrics['ASSD']
    
    return sen, spc, pre, acc, jac, dice, mcc, fmi, brier_score, assd

def calculate_assd(gt_mask, pred_mask):
    """Calculate Average Symmetric Surface Distance and related metrics."""
    # Ensure masks are binary
    gt_binary = (gt_mask > 0.5).astype(np.uint8)
    pred_binary = (pred_mask > 0.5).astype(np.uint8)
    
    def find_boundary_pixels(mask):
        """Find boundary pixels using morphological operations."""
        kernel = np.ones((3, 3), dtype=np.uint8)
        eroded = cv2.erode(mask, kernel)
        dilated = cv2.dilate(mask, kernel)
        boundary = dilated - eroded
        return boundary
    
    def calculate_boundary_distances(mask1, mask2):
        """Calculate distance from each boundary point in mask1 to nearest boundary point in mask2."""
        boundary1 = find_boundary_pixels(mask1)
        boundary2 = find_boundary_pixels(mask2)
        
        # Get coordinates of boundary points
        coords1 = np.column_stack(np.where(boundary1 > 0))
        coords2 = np.column_stack(np.where(boundary2 > 0))
        
        if len(coords1) == 0 or len(coords2) == 0:
            return np.array([]), coords1
        
        # Create distance transform to the second boundary
        boundary2_binary = np.zeros_like(mask2, dtype=bool)
        boundary2_binary[coords2[:, 0], coords2[:, 1]] = True
        dist_transform = ndimage.distance_transform_edt(~boundary2_binary)
        
        # Get distances for each point in boundary1
        distances = dist_transform[coords1[:, 0], coords1[:, 1]]
        
        return distances, coords1
    
    # Calculate distances from GT boundary to Pred boundary
    dist_gt_to_pred, gt_boundary = calculate_boundary_distances(gt_binary, pred_binary)
    dist_pred_to_gt, pred_boundary = calculate_boundary_distances(pred_binary, gt_binary)
    
    if len(dist_gt_to_pred) == 0 or len(dist_pred_to_gt) == 0:
        return {
            'ASSD': np.inf, 
            'ASD_GT_to_Pred': np.inf, 
            'ASD_Pred_to_GT': np.inf,
            'GT_Boundary_Points': len(gt_boundary),
            'Pred_Boundary_Points': len(pred_boundary)
        }
    
    # Calculate average surface distances
    asd_gt_to_pred = np.mean(dist_gt_to_pred)
    asd_pred_to_gt = np.mean(dist_pred_to_gt)
    
    # Symmetric average
    assd = (asd_gt_to_pred + asd_pred_to_gt) / 2
    
    return {
        'ASSD': assd,
        'ASD_GT_to_Pred': asd_gt_to_pred,
        'ASD_Pred_to_GT': asd_pred_to_gt,
        'GT_Boundary_Points': len(gt_boundary),
        'Pred_Boundary_Points': len(pred_boundary)
    }

def calculate_hausdorff(y_true, y_pred):
    # Get contours from masks
    contours_true, _ = cv2.findContours(y_true.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours_pred, _ = cv2.findContours(y_pred.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if len(contours_true) == 0 or len(contours_pred) == 0:
        return 0, 0  # No contours found
    
    # Combine all contour points
    points_true = np.vstack([c.squeeze() for c in contours_true if len(c) > 0])
    points_pred = np.vstack([c.squeeze() for c in contours_pred if len(c) > 0])
    
    if len(points_true) == 0 or len(points_pred) == 0:
        return 0, 0
    
    # Calculate directed average Hausdorff distances
    def directed_avg_hausdorff(points_a, points_b):
        distances = []
        for a in points_a:
            min_dist = np.min(np.sqrt(((points_b - a) ** 2).sum(axis=1)))
            distances.append(min_dist)
        return np.mean(distances)
    
    # Compute average directed distances
    avg_dist_true_to_pred = directed_avg_hausdorff(points_true, points_pred)
    avg_dist_pred_to_true = directed_avg_hausdorff(points_pred, points_true)
    
    # Average Hausdorff distance (h1)
    h1 = max(avg_dist_true_to_pred, avg_dist_pred_to_true)
    
    # Relative Hausdorff distance (h2)
    h2 = (h1 / len(points_true)) * 100
    
    return h1, h2

# Get all image files
img_files = [f for f in os.listdir(gray_dir) if f.lower().endswith(('.jpg', '.jpeg', '.bmp', '.png'))]


#filename = os.path.basename(img_path)
#base_name = os.path.splitext(filename)[0]
#mask_name = f"{base_name}_1stHO.png"
#gt_path = os.path.join(gt_dir, mask_name)
for img_file in img_files:
    # Store image name
    metrics['RVS']['Image'].append(img_file)
    
    # Construct paths
    base_name = os.path.splitext(img_file)[0]
    img_path = os.path.join(gray_dir, img_file)
    rvs_mask_path = os.path.join(rvs_dir, f"{base_name}_1stHO.png")
    
    # Load and process image
    img = cv2.imread(img_path)
    if img is None:
        print(f"Warning: Could not read image {img_path}")
        continue
    
    img = cv2.resize(img, (512, 512))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_input = np.expand_dims(img, 0)[:,:,:,0:3]
    
    # Get predictions
    rvs = model.predict(img_input)
    rvs_prob = rvs[0,:,:,0]
    rvs_pred = (rvs_prob > 0.1).astype(np.uint8)
    
    # Load ground truth mask
    rvs_true = cv2.imread(rvs_mask_path, cv2.IMREAD_GRAYSCALE)
    if rvs_true is None:
        print(f"Warning: Could not read mask for {img_file}")
        continue
    
    rvs_true = cv2.resize(rvs_true, (512, 512))
    _, rvs_true = cv2.threshold(rvs_true, 127, 1, cv2.THRESH_BINARY)
    
    # Calculate metrics for RVS
    sen, spc, pre, acc, jac, dice, mcc, fmi, brier_score, assd = calculate_all_metrics(rvs_true, rvs_pred, rvs_prob)
    h1, h2 = calculate_hausdorff(rvs_true, rvs_pred)
    
    metrics['RVS']['SEN'].append(sen)
    metrics['RVS']['SPC'].append(spc)
    metrics['RVS']['PRE'].append(pre)
    metrics['RVS']['ACC'].append(acc)
    metrics['RVS']['JAC'].append(jac)
    metrics['RVS']['DICE'].append(dice)
    metrics['RVS']['H1'].append(h1)
    metrics['RVS']['H2'].append(h2)
    metrics['RVS']['MCC'].append(mcc)
    metrics['RVS']['FMI'].append(fmi)
    metrics['RVS']['Brier_Score'].append(brier_score)
    metrics['RVS']['ASSD'].append(assd)

# Create DataFrame with image names as the first column
df_rvs = pd.DataFrame(metrics['RVS'])

# Reorder columns to have Image first
cols_rvs = ['Image'] + [col for col in df_rvs.columns if col != 'Image']
df_rvs = df_rvs[cols_rvs]

# Save to CSV
df_rvs.to_csv("CHASE1.csv", index=False)

Model loaded successfully.
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 298ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 316ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 363ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 391ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 354ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 366ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 325ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 359ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 344ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 340ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 339ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 344ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 344ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 348ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 327ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 351ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 349ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 337ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 343ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 356ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 351ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 340ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 340ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 338ms/step
1